# Workflow Charts: Data Acquisition and Data Pre-processing

Draws the process workflow charts for the methodology chapter (one per procedure) and saves each one in three formats to `outputs/figures/`:
- `.png`: 300 dpi, for Word or Google Docs
- `.svg`: vector, with text that stays editable (for example in Inkscape or PowerPoint)
- `.pdf`: vector, for LaTeX

Each chart is laid out at 6.5 in wide, the text width of a page with 1-inch margins, so the text prints at its intended size (7–8 pt).

**To edit the wording:** change the text in a chart cell and run the setup cell and that chart cell again. If the text no longer fits, adjust the box height `h` on the same line.

**Symbols (flowchart conventions):**
- cylinder = external data source
- rectangle = process
- diamond = decision
- wavy-bottom shape = output file
- rounded end = start, end or hand-over to the next stage

In [ ]:
"""
workflow_charts.py

Purpose:
    Draw publication-ready process workflow charts with matplotlib.
    All coordinates are in inches; y increases upwards.

Output:
    outputs/figures/
"""

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Ellipse, PathPatch, FancyArrowPatch, Rectangle
from matplotlib.path import Path as MPath
from matplotlib.transforms import Bbox


# ============================================================
# 1. SETTINGS
# ============================================================

OUTPUT_DIR = Path(
    "outputs/figures"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

plt.rcParams.update({
    "font.family":      "Arial",
    "mathtext.fontset": "custom",       # H$_0$ etc. drawn in Arial
    "mathtext.rm":      "Arial",
    "mathtext.it":      "Arial:italic",
    "mathtext.bf":      "Arial:bold",
    "svg.fonttype":     "none",         # editable text in SVG
    "pdf.fonttype":     42              # editable text in PDF
})

FS_TITLE = 7.6      # bold first line of a box (pt)
FS_BODY = 7.1       # box text (pt)
GAP = 0.13          # vertical space between boxes (in)

EDGE = "#2b2b2b"
TEXT = "#1a1a1a"
NOTE = "#5a5a5a"
WHITE = "#ffffff"
OUTPUT_FILL = "#e9ecef"

LANE_COLORS = {
    "blue":  ("#eef3f8", "#c9d6e3"),
    "green": ("#eef6f0", "#c8dccd"),
    "sand":  ("#f8f3ea", "#e3d6bf")
}


# ============================================================
# 2. DRAWING HELPERS
# ============================================================

class Chart:
    """A blank canvas measured in inches, with flowchart shapes."""

    def __init__(self, width, height):

        self.fig = plt.figure(figsize=(width, height))
        self.ax = self.fig.add_axes([0, 0, 1, 1])
        self.ax.set_xlim(0, width)
        self.ax.set_ylim(0, height)
        self.ax.axis("off")
        self.renderer = self.fig.canvas.get_renderer()

    # ---------------- text ----------------

    def _height(self, artist):

        return artist.get_window_extent(renderer=self.renderer).height / self.fig.dpi

    def label(self, x, y, title=None, body=None, note=None):
        """Centre a bold title, body text and an italic note on (x, y)."""

        parts = []

        if title:
            parts.append(self.ax.text(x, 0, title, ha="center", va="top", fontsize=FS_TITLE,
                                      fontweight="bold", color=TEXT, linespacing=1.2, zorder=4))
        if body:
            parts.append(self.ax.text(x, 0, body, ha="center", va="top", fontsize=FS_BODY,
                                      color=TEXT, linespacing=1.25, zorder=4))
        if note:
            parts.append(self.ax.text(x, 0, note, ha="center", va="top", fontsize=FS_BODY - 0.4,
                                      style="italic", color=NOTE, linespacing=1.2, zorder=4))

        if not parts:
            return

        heights = [self._height(part) for part in parts]
        spacing = 0.035
        top = y + (sum(heights) + spacing * (len(parts) - 1)) / 2

        for part, height in zip(parts, heights):
            part.set_y(top)
            top -= height + spacing

    # ---------------- shapes ----------------

    def _add(self, patch):

        patch.set_zorder(2)
        self.ax.add_patch(patch)

    def process(self, x, y, w, h, title=None, body=None, note=None, dashed=False):

        self._add(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                 boxstyle="round,pad=0,rounding_size=0.05",
                                 fc=WHITE, ec=EDGE, lw=0.8, ls=(0, (4, 2)) if dashed else "-"))
        self.label(x, y, title, body, note)
        return dict(x=x, y=y, w=w, h=h)

    def terminator(self, x, y, w, h, title=None, body=None):

        self._add(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                 boxstyle=f"round,pad=0,rounding_size={h / 2}",
                                 fc=WHITE, ec=EDGE, lw=0.8))
        self.label(x, y, title, body)
        return dict(x=x, y=y, w=w, h=h)

    def decision(self, x, y, w, h, text=None):

        self._add(Polygon([(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
                          closed=True, fc=WHITE, ec=EDGE, lw=0.8))
        self.label(x, y, text)
        return dict(x=x, y=y, w=w, h=h)

    def document(self, x, y, w, h, title=None, body=None):

        a = 0.045
        left, right, top, base = x - w / 2, x + w / 2, y + h / 2, y - h / 2 + a
        vertices = [(left, top), (right, top), (right, base),
                    (x + w / 6, base - 2.4 * a), (x - w / 6, base + 2.4 * a), (left, base),
                    (left, top)]
        codes = [MPath.MOVETO, MPath.LINETO, MPath.LINETO,
                 MPath.CURVE4, MPath.CURVE4, MPath.CURVE4, MPath.CLOSEPOLY]
        self._add(PathPatch(MPath(vertices, codes), fc=OUTPUT_FILL, ec=EDGE, lw=0.8))
        self.label(x, y + 0.02, title, body)
        return dict(x=x, y=y, w=w, h=h)

    def cylinder(self, x, y, w, h, title=None, body=None):

        ry, k, a = 0.055, 0.5523, w / 2
        y_top, y_bottom = y + h / 2 - ry, y - h / 2 + ry
        vertices = [(x - a, y_top), (x - a, y_bottom),
                    (x - a, y_bottom - k * ry), (x - k * a, y_bottom - ry), (x, y_bottom - ry),
                    (x + k * a, y_bottom - ry), (x + a, y_bottom - k * ry), (x + a, y_bottom),
                    (x + a, y_top), (x - a, y_top)]
        codes = [MPath.MOVETO, MPath.LINETO,
                 MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
                 MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
                 MPath.LINETO, MPath.CLOSEPOLY]
        self._add(PathPatch(MPath(vertices, codes), fc=WHITE, ec=EDGE, lw=0.8))
        self._add(Ellipse((x, y_top), 2 * a, 2 * ry, fc=WHITE, ec=EDGE, lw=0.8))
        self.label(x, y - ry / 2, title, body)
        return dict(x=x, y=y, w=w, h=h)

    # ---------------- connectors and lanes ----------------

    def arrow(self, *points, head=True):
        """Polyline through the points, with an arrowhead on the last segment."""

        xs, ys = zip(*points)

        if not head:
            self.ax.plot(xs, ys, color=EDGE, lw=0.8, zorder=1)
            return

        if len(points) > 2:
            self.ax.plot(xs[:-1], ys[:-1], color=EDGE, lw=0.8, zorder=1)

        self.ax.add_patch(FancyArrowPatch(points[-2], points[-1], arrowstyle="-|>",
                                          mutation_scale=8, color=EDGE, lw=0.8,
                                          shrinkA=0, shrinkB=0, zorder=1))

    def tag(self, x, y, text, ha="left", boxed=False):
        """Small italic label on a connector."""

        self.ax.text(x, y, text, ha=ha, va="center", fontsize=FS_BODY - 0.3, style="italic",
                     color=TEXT, zorder=5,
                     bbox=dict(fc="white", ec="none", pad=1.0) if boxed else None)

    def lane(self, x0, y0, x1, y1, letter, title, color):

        fill, edge = LANE_COLORS[color]
        self.ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0,
                                         boxstyle="round,pad=0,rounding_size=0.08",
                                         fc=fill, ec=edge, lw=0.8, zorder=0))
        self.ax.text(x0 + 0.12, y1 - 0.08, letter, ha="left", va="top", fontsize=FS_TITLE + 0.6,
                     fontweight="bold", color=TEXT, zorder=4)
        self.ax.text(x0 + 0.34, y1 - 0.08, title, ha="left", va="top", fontsize=FS_TITLE + 0.6,
                     fontweight="bold", color=TEXT, linespacing=1.15, zorder=4)

    def legend(self, y, items):

        names = {
            "cylinder":   "External data source",
            "process":    "Process",
            "decision":   "Decision",
            "document":   "Data file",
            "terminator": "Start / end"
        }

        x = 0.2

        for item in items:

            w, h = 0.34, 0.2
            cx = x + w / 2

            if item == "cylinder":
                self.cylinder(cx, y, w, h + 0.04)
            elif item == "process":
                self.process(cx, y, w, h)
            elif item == "decision":
                self.decision(cx, y, w, h + 0.06)
            elif item == "document":
                self.document(cx, y, w, h + 0.04)
            else:
                self.terminator(cx, y, w, h)

            self.ax.text(x + w + 0.07, y, names[item], ha="left", va="center",
                         fontsize=FS_BODY, color=TEXT)
            x += w + 0.07 + len(names[item]) * 0.055 + 0.28

    def save(self, name):
        """Save PNG (300 dpi), SVG and PDF, cropped to the drawn content."""

        self.fig.canvas.draw()

        artists = self.ax.patches + self.ax.texts + self.ax.lines

        content = Bbox.union(
            [artist.get_window_extent(self.renderer) for artist in artists]
        ).transformed(self.fig.dpi_scale_trans.inverted()).padded(0.04)

        for extension in ("png", "svg", "pdf"):
            self.fig.savefig(OUTPUT_DIR / f"{name}.{extension}", dpi=300,
                             bbox_inches=content, facecolor="white")

        print(f"Saved: {OUTPUT_DIR / name}.png / .svg / .pdf")


def top(s):
    return (s["x"], s["y"] + s["h"] / 2)


def bottom(s):
    return (s["x"], s["y"] - s["h"] / 2)


def left(s):
    return (s["x"] - s["w"] / 2, s["y"])


def right(s):
    return (s["x"] + s["w"] / 2, s["y"])



# ============================================================
# 3. EXTRA HELPERS (forks, joins, loops, timelines)
# ============================================================

TRAIN_FILL = "#9ec5f4"     # training years (blue)
TEST_FILL = "#ee8156"      # forecast / validation years (orange)


def fork(c, start, tops, line_y):
    """One line splitting into several arrows (one per branch)."""

    c.arrow(start, (start[0], line_y), head=False)

    xs = [point[0] for point in tops] + [start[0]]

    c.arrow((min(xs), line_y), (max(xs), line_y), head=False)

    for point in tops:
        c.arrow((point[0], line_y), point)


def join(c, bottoms, end, line_y):
    """Several branches merging into one arrow."""

    for point in bottoms:
        c.arrow(point, (point[0], line_y), head=False)

    xs = [point[0] for point in bottoms] + [end[0]]

    c.arrow((min(xs), line_y), (max(xs), line_y), head=False)

    c.arrow((end[0], line_y), end)


def side_label(c, x, y, text):
    """Italic label written along a vertical connector."""

    c.ax.text(x, y, text, rotation=90, ha="center", va="center", fontsize=FS_BODY - 0.4,
              style="italic", color=TEXT, zorder=5, bbox=dict(fc="white", ec="none", pad=1.0))


def timeline(c, x0, y_top, width, rounds, first_year, last_year, ticks, title, test_label):
    """Rows of year cells: training years in blue, the forecast year in orange."""

    cell = width / (last_year - first_year + 1)

    row_h, row_gap = 0.16, 0.07

    c.ax.text(x0, y_top, title, ha="left", va="top", fontsize=FS_TITLE, fontweight="bold", color=TEXT)

    y = y_top - 0.28

    for label, train_end, test_year in rounds:

        c.ax.text(x0 - 0.06, y - row_h / 2, label, ha="right", va="center", fontsize=FS_BODY - 0.4, color=TEXT)

        for year in range(first_year, last_year + 1):

            if year <= train_end:
                fill = TRAIN_FILL
            elif year == test_year:
                fill = TEST_FILL
            else:
                continue

            c.ax.add_patch(Rectangle((x0 + (year - first_year) * cell, y - row_h), cell, row_h,
                                     fc=fill, ec="white", lw=0.5, zorder=2))

        y -= row_h + row_gap

    for year in ticks:
        c.ax.text(x0 + (year - first_year + 0.5) * cell, y + row_gap - 0.04, str(year),
                  ha="center", va="top", fontsize=FS_BODY - 0.6, color=TEXT)

    legend_y = y - 0.22

    for offset, (fill, text) in enumerate([(TRAIN_FILL, "Training years"), (TEST_FILL, test_label)]):

        lx = x0 + offset * width / 2

        c.ax.add_patch(Rectangle((lx, legend_y - 0.06), 0.16, 0.12, fc=fill, ec="none", zorder=2))

        c.ax.text(lx + 0.22, legend_y, text, ha="left", va="center", fontsize=FS_BODY - 0.4, color=TEXT)

    return legend_y

## Figure 1. Data acquisition workflow

In [ ]:
# ============================================================
# FIGURE 1. DATA ACQUISITION
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.46; A1 = c.cylinder(X, y - h / 2, 3.3, h, "Philippine boundary shapefile",
                          "PH_Adm0_Country · 3,640 polygons · EPSG:32651"); y -= h + GAP

h = 0.48; A2 = c.process(X, y - h / 2, 4.9, h, "Prepare the land boundary",
                         "Reproject to WGS 84 (EPSG:4326) · validate geometries · "
                         "dissolve into one land polygon"); y -= h + GAP

h = 0.40; A3 = c.process(X, y - h / 2, 4.9, h, "Generate candidate grid",
                         "1° × 1° cells covering 116–128° E and 4–22° N (216 cells)"); y -= h + GAP

h = 0.60; A4 = c.decision(X, y - h / 2, 2.1, h, "Cell intersects\nPhilippine land?"); y -= h + GAP

DISCARD = c.terminator(X + 2.2, A4["y"], 1.4, 0.34, "Discard", "123 cells")

h = 0.62; A5 = c.process(X, y - h / 2, 5.5, h, "Characterise the retained cells",
                         "Number cells 1–93 (south to north, west to east) · cell and land area in an\n"
                         "equal-area projection (EPSG:6933) · land coverage (%) · a representative\n"
                         "point inside each cell's land area, used as the query coordinates"); y -= h + GAP

h = 0.44; A6 = c.document(X, y - h / 2, 3.9, h, "Spatial grid (93 cells)",
                          "grid_metadata.csv · philippines_grid.geojson"); y -= h

c.arrow(bottom(A1), top(A2))
c.arrow(bottom(A2), top(A3))
c.arrow(bottom(A3), top(A4))
c.arrow(right(A4), left(DISCARD))
c.tag((right(A4)[0] + left(DISCARD)[0]) / 2, A4["y"] + 0.09, "No", ha="center")
c.arrow(bottom(A4), top(A5))
c.tag(X + 0.08, (bottom(A4)[1] + top(A5)[1]) / 2, "Yes (93 cells)")
c.arrow(bottom(A5), top(A6))


# ------------------------------------------------------------
# Two branches fed by the 93 query coordinates
# ------------------------------------------------------------

XB, XC = 1.62, W - 1.62

fork_y = y - 0.17

y -= 0.36

y_b = y_c = y

h = 0.46; B1 = c.cylinder(XB, y_b - h / 2, 2.6, h, "NASA POWER API",
                          "Monthly point data · RE community"); y_b -= h + GAP

h = 0.78; B2 = c.process(XB, y_b - h / 2, 2.85, h, "Request variables per query point",
                         "ALLSKY_SFC_SW_DWN → GHI (kWh/m²/day)\n"
                         "CLOUD_AMT → cloud amount (%)\n"
                         "RH2M → relative humidity at 2 m (%)\n"
                         "January 2005 – December 2024"); y_b -= h + GAP

h = 0.48; B3 = c.process(XB, y_b - h / 2, 2.85, h, "Compile and check",
                         "Parse into monthly records ·\n240 months per grid, no gaps"); y_b -= h + GAP

h = 0.46; B4 = c.document(XB, y_b - h / 2, 2.75, h, "nasa_power_monthly.csv",
                          "93 grids × 240 months = 22,320 records"); y_b -= h

h = 0.46; C1 = c.cylinder(XC, y_c - h / 2, 2.6, h, "OpenTopoData API",
                          "Site elevation: NASA SRTM GL1 v3 (30 m)"); y_c -= h + GAP

h = 0.56; C2 = c.decision(XC, y_c - h / 2, 1.7, h, "SRTM value\navailable?"); y_c -= h + GAP

h = 0.48; C3 = c.process(XC, y_c - h / 2, 2.2, h, "Fill the SRTM void",
                         "SRTM 90 m (CGIAR v4.1), then\nASTER GDEM v3 (30 m)")

C4 = c.document(XC, B4["y"], 2.75, 0.46, "grid_elevation.csv",
                "92 cells SRTM GL1 · 1 cell ASTER GDEM")

c.arrow(bottom(A6), (X, fork_y), head=False)
c.arrow((X, fork_y), (XB, fork_y), top(B1))
c.arrow((X, fork_y), (XC, fork_y), top(C1))
c.tag(X, fork_y, "93 query coordinates", ha="center", boxed=True)

c.arrow(bottom(B1), top(B2))
c.arrow(bottom(B2), top(B3))
c.arrow(bottom(B3), top(B4))

c.arrow(bottom(C1), top(C2))
c.arrow(bottom(C2), top(C3))
c.tag(XC + 0.07, (bottom(C2)[1] + top(C3)[1]) / 2, "No (1 cell)")
c.arrow(bottom(C3), top(C4))

yes_x = W - 0.12
yes_y = top(C4)[1] + 0.12
c.arrow(right(C2), (yes_x, C2["y"]), (yes_x, yes_y), (XC + 0.9, yes_y), (XC + 0.9, top(C4)[1]))
c.tag(right(C2)[0] + 0.05, C2["y"] + 0.1, "Yes (92 cells)")


# ------------------------------------------------------------
# Join -> next stage, legend
# ------------------------------------------------------------

join_y = min(bottom(B4)[1], bottom(C4)[1]) - 0.16

END = c.terminator(X, join_y - 0.16 - 0.16, 2.3, 0.32, "To data pre-processing")

join(c, [bottom(B4), bottom(C4)], top(END), join_y)

c.legend(END["y"] - 0.42, ["cylinder", "process", "decision", "document", "terminator"])

c.save("data_acquisition_workflow")

plt.show()

## Figure 2. Data pre-processing workflow

In [ ]:
# ============================================================
# FIGURE 2. DATA PRE-PROCESSING
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

BW = 5.6

h = 0.46; IN = c.document(X, y - h / 2, 4.6, h, "Acquired data",
                          "nasa_power_monthly.csv · grid_metadata.csv · grid_elevation.csv"); y -= h + GAP

h = 0.56; P1 = c.process(X, y - h / 2, BW, h, "Structural checks",
                         "Required columns and types · 93 grids match the grid metadata · 240 months per grid ·\n"
                         "no duplicate grid–month records · continuous monthly sequence, Jan 2005 – Dec 2024"); y -= h + GAP

h = 0.42; P2 = c.process(X, y - h / 2, BW, h, "Missing values",
                         "Empty values and the NASA POWER fill value (−999) in GHI, cloud amount and RH"); y -= h + GAP

h = 0.56; P3 = c.process(X, y - h / 2, BW, h, "Physical validity",
                         "Extraterrestrial irradiation H$_0$ and clearness index Kt = GHI / H$_0$ for every grid and month ·\n"
                         "GHI invalid if < 0 or Kt > 1; implausible if Kt is outside 0.10–0.80 · cloud amount and RH within 0–100 %"); y -= h + GAP

h = 0.42; P4 = c.process(X, y - h / 2, BW, h, "Statistical outliers",
                         "Modified z-score within each grid and calendar month; |z| > 3.5 flagged and retained"); y -= h + GAP

h = 0.56; P5 = c.process(X, y - h / 2, BW, h, "Consistency checks",
                         "Flat lines (≥ 3 identical consecutive months) · negative GHI–cloud correlation\n"
                         "in every grid · elevation available for every grid"); y -= h + GAP

h = 0.6; DQ = c.decision(X, y - h / 2, 1.8, h, "Any FAIL-level\nissue?")

FIX = c.process(X + 2.05, DQ["y"], 1.75, 0.56, "Resolve the issue",
                "Re-extract or correct\nthe affected records")

y -= h + GAP

h = 0.5; OUT = c.document(X, y - h / 2, 4.9, h, "Validated dataset",
                          "nasa_power_validated.csv (adds H$_0$, Kt and quality flags) ·\n"
                          "validation_summary.csv · flagged_records.csv"); y -= h + GAP

h = 0.42; Q1 = c.process(X, y - h / 2, BW, h, "Merge and encode",
                         "join SRTM elevation to every grid cell · encode the month as sine and cosine"); y -= h + GAP

h = 0.42; Q2 = c.process(X, y - h / 2, BW, h, "Chronological split",
                         "2005–2022: model development and rolling validation (8 rounds) · 2023–2024: final testing"); y -= h + GAP

h = 0.42; Q3 = c.process(X, y - h / 2, BW, h, "Training-only transforms (each round)",
                         "min–max scaling fitted on the training rows · 12-month input windows from past values only"); y -= h

c.arrow(bottom(IN), top(P1))

for upper, lower in [(P1, P2), (P2, P3), (P3, P4), (P4, P5), (P5, DQ)]:
    c.arrow(bottom(upper), top(lower))

c.arrow(right(DQ), left(FIX))
c.tag((right(DQ)[0] + left(FIX)[0]) / 2, DQ["y"] + 0.09, "Yes", ha="center")

loop_x = W - 0.12

c.arrow(right(FIX), (loop_x, FIX["y"]), (loop_x, P1["y"]), right(P1))

side_label(c, loop_x, (FIX["y"] + P1["y"]) / 2, "re-run all checks")

c.arrow(bottom(DQ), top(OUT))
c.tag(X + 0.08, (bottom(DQ)[1] + top(OUT)[1]) / 2, "No")

c.arrow(bottom(OUT), top(Q1))
c.arrow(bottom(Q1), top(Q2))
c.arrow(bottom(Q2), top(Q3))

y -= 0.18

END = c.terminator(X, y - 0.16, 2.6, 0.32, "To model development")

c.arrow(bottom(Q3), top(END))

c.legend(END["y"] - 0.42, ["process", "decision", "document", "terminator"])

c.save("data_preprocessing_workflow")

plt.show()

## Figure 3. Development of standalone models

In [ ]:
# ============================================================
# FIGURE 3. DEVELOPMENT OF STANDALONE MODELS
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.44; IN = c.document(X, y - h / 2, 4.4, h, "Prepared data (2005–2022)",
                          "Monthly GHI, cloud amount and RH · elevation · 93 grid cells"); y -= h + 0.2

h = 0.52; TASK = c.process(X, y - h / 2, 5.6, h, "Define the forecast task (the same for every model)",
                           "From each December origin, forecast the 12 monthly GHI values of the next year,\n"
                           "using only data available at the origin"); y -= h + 0.32

BW, BH = 1.48, 1.02

xs = [0.23 + BW / 2 + i * (BW + 0.08) for i in range(4)]

M1 = c.process(xs[0], y - BH / 2, BW, BH, "SARIMAX",
               "One model per grid cell\nPast GHI + cloud\namount and RH of\n12 months earlier")
M2 = c.process(xs[1], y - BH / 2, BW, BH, "LSTM",
               "One global model\n12-month input window\n2 layers × 64 units\n12 outputs at once")
M3 = c.process(xs[2], y - BH / 2, BW, BH, "XGBoost",
               "One global model\nPast 12 months of GHI,\ncloud amount and RH\n+ horizon and month")
M4 = c.process(xs[3], y - BH / 2, BW, BH, "Reference forecasts",
               "Climatology:\nmonthly average\nSeasonal naive:\nsame month last year")

fork(c, bottom(TASK), [top(M) for M in (M1, M2, M3, M4)], bottom(TASK)[1] - 0.16)

y -= BH + 0.3

h = 0.46; OUT = c.document(X, y - h / 2, 4.4, h, "Forecasts",
                           "12 monthly GHI values per model, grid cell and forecast year")

join(c, [bottom(M) for M in (M1, M2, M3, M4)], top(OUT), bottom(M1)[1] - 0.15)

c.arrow(bottom(IN), top(TASK))

y -= h + 0.18

END = c.terminator(X, y - 0.16, 3.2, 0.32, "To model training and rolling validation")

c.arrow(bottom(OUT), top(END))

c.legend(END["y"] - 0.42, ["process", "document", "terminator"])

c.save("standalone_models_workflow")

plt.show()

## Figure 4. Development of the hybrid SARIMAX-LSTM

In [ ]:
# ============================================================
# FIGURE 4. DEVELOPMENT OF THE HYBRID SARIMAX-LSTM
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.44; IN = c.document(X, y - h / 2, 4.4, h, "Prepared data",
                          "Monthly GHI, cloud amount and RH · 93 grid cells"); y -= h + 0.32

BW, BH = 2.6, 0.58

S = c.process(X - 1.5, y - BH / 2, BW, BH, "SARIMAX component", "Forecasts GHI on its own\n(one model per grid cell)")
L = c.process(X + 1.5, y - BH / 2, BW, BH, "LSTM component", "Forecasts GHI on its own\n(one global model)")

fork(c, bottom(IN), [top(S), top(L)], bottom(IN)[1] - 0.16)

y -= BH + 0.3

h = 0.56; E = c.process(X, y - h / 2, 5.4, h, "Measure each component's past errors",
                        "Squared errors of its forecasts in all earlier years, per grid cell;\n"
                        "recent years count more (each year further back × 0.8)")

join(c, [bottom(S), bottom(L)], top(E), bottom(S)[1] - 0.15)

y -= h + GAP

h = 0.56; WT = c.process(X, y - h / 2, 5.4, h, "Set time-varying weights (per grid cell and year)",
                         "w(SARIMAX) = error(LSTM) ÷ [error(SARIMAX) + error(LSTM)],  w(LSTM) = 1 − w(SARIMAX)\n"
                         "so the component that has been more accurate recently gets more weight"); y -= h + GAP

h = 0.46; CMB = c.process(X, y - h / 2, 5.4, h, "Combine the two forecasts",
                          "Hybrid forecast = w(SARIMAX) × SARIMAX + w(LSTM) × LSTM"); y -= h + GAP

h = 0.44; OUT = c.document(X, y - h / 2, 3.8, h, "Hybrid SARIMAX-LSTM forecast",
                           "12 monthly GHI values per grid cell"); y -= h

c.arrow(bottom(E), top(WT))
c.arrow(bottom(WT), top(CMB))
c.arrow(bottom(CMB), top(OUT))

loop_x = W - 0.22

c.arrow(right(OUT), (loop_x, OUT["y"]), (loop_x, E["y"]), right(E))

side_label(c, loop_x, (OUT["y"] + E["y"]) / 2, "each new year adds its errors")

y -= 0.18

END = c.terminator(X, y - 0.16, 3.4, 0.32, "Next: add the clear-sky index (stack 2)")

c.arrow(bottom(OUT), top(END))

c.legend(END["y"] - 0.42, ["process", "document", "terminator"])

c.save("hybrid_sarimax_lstm_workflow")

plt.show()

## Figure 5. Incorporation of the Ineichen-Perez clear-sky framework

In [ ]:
# ============================================================
# FIGURE 5. INCORPORATION OF THE INEICHEN-PEREZ CLEAR-SKY FRAMEWORK
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

LX, RX, CW = 1.85, 4.85, 2.8

h = 0.5; C1 = c.cylinder(LX, y - h / 2, CW, h, "Grid points + SRTM elevation",
                         "93 query coordinates and site altitudes")

OBS = c.document(RX, y - h / 2, CW, 0.46, "Observed monthly GHI", "NASA POWER, 2005 onward")

y -= h + GAP

h = 0.82; CS = c.process(LX, y - h / 2, CW, h, "Ineichen-Perez clear-sky GHI",
                         "pvlib: sun position every hour\n+ Linke turbidity climatology\n"
                         "→ daily totals → monthly means\n(2005–2026)"); y -= h + GAP

h = 0.46; CT = c.document(LX, y - h / 2, CW, h, "Clear-sky GHI table", "known in advance for any year"); y -= h + 0.3

h = 0.56; CSI = c.process(X, y - h / 2, 5.4, h, "Clear-sky index (CSI)",
                          "CSI = observed GHI ÷ clear-sky GHI (about 0.4–1.0). It removes the regular seasonal\n"
                          "effect of the sun and leaves the cloud-driven part for the models to learn")

join(c, [bottom(CT), bottom(OBS)], top(CSI), bottom(CT)[1] - 0.15)

c.arrow(bottom(C1), top(CS))
c.arrow(bottom(CS), top(CT))

y -= h + GAP

h = 0.46; FC = c.process(X, y - h / 2, 5.4, h, "SARIMAX and LSTM forecast the CSI",
                         "The same two components as the hybrid, with CSI in place of GHI"); y -= h + GAP

h = 0.46; BACK = c.process(X, y - h / 2, 5.4, h, "Convert back to GHI",
                           "Forecast GHI = forecast CSI × clear-sky GHI of the target month"); y -= h + GAP

h = 0.46; CMB = c.process(X, y - h / 2, 5.4, h, "Combine with time-varying weights",
                          "The same weighting as the hybrid SARIMAX-LSTM"); y -= h + GAP

h = 0.44; OUT = c.document(X, y - h / 2, 3.8, h, "Hybrid + CSI forecast",
                           "12 monthly GHI values per grid cell"); y -= h

c.arrow(bottom(CSI), top(FC))
c.arrow(bottom(FC), top(BACK))
c.arrow(bottom(BACK), top(CMB))
c.arrow(bottom(CMB), top(OUT))

route_x = 0.2

c.arrow(left(CT), (route_x, CT["y"]), (route_x, BACK["y"]), left(BACK))

side_label(c, route_x, (CT["y"] + BACK["y"]) / 2, "clear-sky GHI of the target month")

y -= 0.18

END = c.terminator(X, y - 0.16, 3.4, 0.32, "Next: add the penalized loss (stack 3)")

c.arrow(bottom(OUT), top(END))

c.legend(END["y"] - 0.42, ["cylinder", "process", "document", "terminator"])

c.save("clear_sky_framework_workflow")

plt.show()

## Figure 6. Integration of the penalized loss function

In [ ]:
# ============================================================
# FIGURE 6. INTEGRATION OF THE PENALIZED LOSS FUNCTION
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.46; IN = c.document(X, y - h / 2, 4.6, h, "Training samples (clear-sky index)",
                          "12 months of CSI, cloud amount and RH → the next 12 CSI values"); y -= h + GAP

h = 0.44; P1 = c.process(X, y - h / 2, 4.6, h, "The LSTM forecasts 12 CSI values",
                         "for a batch of training samples"); y -= h + 0.32

BW, BH = 1.36, 0.86

start = (W - (4 * BW + 3 * 0.08)) / 2

xs = [start + BW / 2 + i * (BW + 0.08) for i in range(4)]

T1 = c.process(xs[0], y - BH / 2, BW, BH, "Accuracy",
               "Mean squared error\nbetween forecast and\nobserved CSI")
T2 = c.process(xs[1], y - BH / 2, BW, BH, "Clear-sky penalty",
               "Weight 10: forecast\nCSI above 1 (GHI above\nclear-sky) or below 0")
T3 = c.process(xs[2], y - BH / 2, BW, BH, "Inaccuracy penalty",
               "Weight 1: errors\nlarger than 0.05 CSI")
T4 = c.process(xs[3], y - BH / 2, BW, BH, "Accuracy reward",
               "Weight −0.005: close\nto observed and\nphysically valid")

fork(c, bottom(P1), [top(T) for T in (T1, T2, T3, T4)], bottom(P1)[1] - 0.16)

y -= BH + 0.3

h = 0.46; LOSS = c.process(X, y - h / 2, 4.6, h, "Total loss = the four terms combined",
                           "The network's weights are updated to reduce it (Adam optimiser)")

join(c, [bottom(T) for T in (T1, T2, T3, T4)], top(LOSS), bottom(T1)[1] - 0.15)

y -= h + GAP

h = 0.86; DEC = c.decision(X, y - h / 2, 2.7, h, "Early-stopping loss\nimproved in the last\n20 epochs?"); y -= h + GAP

h = 0.44; BEST = c.process(X, y - h / 2, 4.6, h, "Keep the weights from the best epoch",
                           "This is the trained LSTM (CSI + PL) component"); y -= h + GAP

h = 0.46; CMB = c.process(X, y - h / 2, 4.6, h, "Combine with the SARIMAX (CSI) component",
                          "time-varying weights, as in the hybrid"); y -= h + GAP

h = 0.44; OUT = c.document(X, y - h / 2, 4.0, h, "Proposed model",
                           "Hybrid SARIMAX-LSTM + CSI + PL"); y -= h

c.arrow(bottom(IN), top(P1))
c.arrow(bottom(LOSS), top(DEC))
c.arrow(bottom(DEC), top(BEST))
c.tag(X + 0.08, (bottom(DEC)[1] + top(BEST)[1]) / 2, "No: stop")
c.arrow(bottom(BEST), top(CMB))
c.arrow(bottom(CMB), top(OUT))

loop_x = W - 0.18

c.arrow(right(DEC), (loop_x, DEC["y"]), (loop_x, P1["y"]), right(P1))

c.tag(right(DEC)[0] + 0.06, DEC["y"] + 0.1, "Yes")

side_label(c, loop_x, (DEC["y"] + P1["y"]) / 2, "next epoch")

y -= 0.18

END = c.terminator(X, y - 0.16, 3.4, 0.32, "To model training and rolling validation")

c.arrow(bottom(OUT), top(END))

c.legend(END["y"] - 0.42, ["process", "decision", "document", "terminator"])

c.save("penalized_loss_workflow")

plt.show()

## Figure 7. Model training and rolling validation

In [ ]:
# ============================================================
# FIGURE 7. MODEL TRAINING AND ROLLING VALIDATION
# ============================================================

W = 6.5
c = Chart(W, 10.0)
y = 9.95

FX, FW = 1.85, 3.0

h = 0.44; IN = c.document(FX, y - h / 2, FW, h, "Development data",
                          "2005–2022 (2023–2024 held out)"); y -= h + GAP

h = 0.5; TR = c.process(FX, y - h / 2, FW, h, "Train every model",
                        "on 2005 to year Y − 1\n(round 1: Y = 2015)"); y -= h + GAP

h = 0.5; FC = c.process(FX, y - h / 2, FW, h, "Forecast year Y",
                        "12 monthly values from the\nDecember origin"); y -= h + GAP

h = 0.5; SC = c.process(FX, y - h / 2, FW, h, "Score against observations",
                        "RMSE, MAE and nRMSE\nper grid cell"); y -= h + GAP

h = 0.6; DQ = c.decision(FX, y - h / 2, 1.9, h, "Y = 2022?"); y -= h + GAP

h = 0.44; OUT = c.document(FX, y - h / 2, FW, h, "Rolling validation results",
                           "by round and grid cell, all models"); y -= h

c.arrow(bottom(IN), top(TR))
c.arrow(bottom(TR), top(FC))
c.arrow(bottom(FC), top(SC))
c.arrow(bottom(SC), top(DQ))
c.arrow(bottom(DQ), top(OUT))
c.tag(FX + 0.08, (bottom(DQ)[1] + top(OUT)[1]) / 2, "Yes")

loop_x = 0.14

c.arrow(left(DQ), (loop_x, DQ["y"]), (loop_x, TR["y"]), left(TR))

c.tag(left(DQ)[0] - 0.06, DQ["y"] + 0.1, "No", ha="right")

side_label(c, loop_x, (DQ["y"] + TR["y"]) / 2, "next round: Y + 1")

# Timeline of the 8 expanding-window rounds
timeline(
    c, x0=4.05, y_top=top(IN)[1], width=2.35,
    rounds=[(f"Round {r}", 2013 + r, 2014 + r) for r in range(1, 9)],
    first_year=2005, last_year=2022, ticks=[2005, 2015, 2022],
    title="Expanding window (8 rounds)", test_label="Validation year"
)

y -= 0.18

END = c.terminator(FX, y - 0.16, FW, 0.32, "To final model testing")

c.arrow(bottom(OUT), top(END))

c.legend(END["y"] - 0.42, ["process", "decision", "document", "terminator"])

c.save("rolling_validation_workflow")

plt.show()

## Figure 8. Final model testing

In [ ]:
# ============================================================
# FIGURE 8. FINAL MODEL TESTING
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.44; IN = c.document(X, y - h / 2, 4.4, h, "All data 2005–2024",
                          "the test years 2023–2024 are used for the first time"); y -= h + GAP

h = 0.46; FIX = c.process(X, y - h / 2, 5.4, h, "Keep every setting from model development",
                          "same model orders, architectures, loss and weighting · no re-tuning"); y -= h + 0.32

BW, BH = 2.6, 0.58

R1 = c.process(X - 1.45, y - BH / 2, BW, BH, "Round 1", "Train on 2005–2022\nForecast 2023 (12 months)")
R2 = c.process(X + 1.45, y - BH / 2, BW, BH, "Round 2", "Retrain on 2005–2023\nForecast 2024 (12 months)")

fork(c, bottom(FIX), [top(R1), top(R2)], bottom(FIX)[1] - 0.16)

y -= BH + 0.3

h = 0.46; POOL = c.process(X, y - h / 2, 5.4, h, "Pool all test forecasts",
                           "93 grid cells × 24 months = 2,232 forecasts per model")

join(c, [bottom(R1), bottom(R2)], top(POOL), bottom(R1)[1] - 0.15)

y -= h + GAP

h = 0.46; MET = c.process(X, y - h / 2, 5.4, h, "Compute pooled metrics",
                          "RMSE, MAE, nRMSE and R² (coefficient of determination); skill vs climatology"); y -= h + GAP

h = 0.44; OUT = c.document(X, y - h / 2, 4.0, h, "Final test results",
                           "overall and per grid cell, all models"); y -= h

c.arrow(bottom(IN), top(FIX))
c.arrow(bottom(POOL), top(MET))
c.arrow(bottom(MET), top(OUT))

y -= 0.18

END = c.terminator(X, y - 0.16, 4.2, 0.32, "To spatial analysis and the comparative analysis")

c.arrow(bottom(OUT), top(END))

c.legend(END["y"] - 0.42, ["process", "document", "terminator"])

c.save("final_testing_workflow")

plt.show()

## Figure 9. Spatial forecasting analysis and mapping

In [ ]:
# ============================================================
# FIGURE 9. SPATIAL FORECASTING ANALYSIS AND MAPPING
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.5

D1 = c.document(X - 1.5, y - h / 2, 2.7, h, "Forecasts and metrics",
                "per grid cell: final test, significance\ntests and the 2026 forecast")
D2 = c.document(X + 1.5, y - h / 2, 2.7, h, "Grid polygons and boundary",
                "93 grid cells · Philippine outline")

y -= h + 0.3

h = 0.44; JN = c.process(X, y - h / 2, 5.4, h, "Join the values to the 93 grid cells",
                         "matched by grid number (for the maps)")

join(c, [bottom(D1), bottom(D2)], top(JN), bottom(D1)[1] - 0.15)

y -= h + GAP

h = 0.46; COL = c.process(X, y - h / 2, 5.4, h, "Choose one colour scale per figure",
                          "blue = error · blue–red = difference · orange = GHI"); y -= h + 0.32

BW, BH = 1.36, 0.86

start = (W - (4 * BW + 3 * 0.08)) / 2

xs = [start + BW / 2 + i * (BW + 0.08) for i in range(4)]

F1 = c.process(xs[0], y - BH / 2, BW, BH, "R² graph",
               "Forecast vs observed\nGHI (final test), with\nthe 1:1 line (no map)")
F2 = c.process(xs[1], y - BH / 2, BW, BH, "RMSE maps",
               "Pooled RMSE per\ngrid cell (final test),\nsix models")
F3 = c.process(xs[2], y - BH / 2, BW, BH, "Difference maps",
               "Proposed minus each\nbenchmark (2015–2024),\nsignificant cells marked")
F4 = c.process(xs[3], y - BH / 2, BW, BH, "2026 forecast maps",
               "Annual map (two\nunits) and 12\nmonthly maps")

fork(c, bottom(COL), [top(F) for F in (F1, F2, F3, F4)], bottom(COL)[1] - 0.16)

y -= BH + 0.3

h = 0.44; OUT = c.document(X, y - h / 2, 4.0, h, "Figures", "PNG (300 dpi), SVG and PDF")

join(c, [bottom(F) for F in (F1, F2, F3, F4)], top(OUT), bottom(F1)[1] - 0.15)

c.arrow(bottom(JN), top(COL))

y -= h

c.legend(y - 0.4, ["process", "document"])

c.save("spatial_mapping_workflow")

plt.show()

## Figure 10. Comparative analysis and ablation analysis

In [ ]:
# ============================================================
# FIGURE 10. COMPARATIVE ANALYSIS AND ABLATION ANALYSIS
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.46; IN = c.document(X, y - h / 2, 4.6, h, "Out-of-sample forecasts, 2015–2024",
                          "rolling validation + final test · 11,160 per model"); y -= h + GAP

h = 0.42; SE = c.process(X, y - h / 2, 5.6, h, "Squared error of every forecast",
                         "error² = (forecast − observed)²"); y -= h + GAP

h = 0.56; LD = c.process(X, y - h / 2, 5.6, h, "Monthly loss differential (model A vs model B)",
                         "d = average over the 93 grid cells of [error² of A − error² of B],\n"
                         "one value for each of the 120 months"); y -= h + GAP

h = 0.46; DM = c.process(X, y - h / 2, 5.6, h, "Diebold–Mariano test with HLN correction",
                         "allows for correlation up to 11 months apart · Student's t, df = 119 · two-sided"); y -= h + 0.32

BW, BH = 2.7, 0.86

A = c.process(X - 1.45, y - BH / 2, BW, BH, "Comparative analysis",
              "Proposed model vs SARIMAX,\nLSTM, XGBoost, climatology\nand seasonal naive\n(5 tests)")
B = c.process(X + 1.45, y - BH / 2, BW, BH, "Ablation analysis",
              "Hybrid vs Hybrid + CSI\nHybrid + CSI vs Hybrid + CSI + PL\nHybrid vs Hybrid + CSI + PL\n(3 tests)")

fork(c, bottom(DM), [top(A), top(B)], bottom(DM)[1] - 0.16)

y -= BH + 0.3

h = 0.56; DEC = c.process(X, y - h / 2, 5.6, h, "Holm adjustment and decision (α = 0.05)",
                          "significant if the adjusted p < 0.05;\n"
                          "the sign of d shows which model is more accurate")

join(c, [bottom(A), bottom(B)], top(DEC), bottom(A)[1] - 0.15)

y -= h + GAP

h = 0.44; OUT = c.document(X, y - h / 2, 4.2, h, "Test tables",
                           "comparative and ablation results"); y -= h

c.arrow(bottom(IN), top(SE))
c.arrow(bottom(SE), top(LD))
c.arrow(bottom(LD), top(DM))
c.arrow(bottom(DEC), top(OUT))

c.legend(y - 0.4, ["process", "document"])

c.save("comparative_ablation_workflow")

plt.show()

## Figure 11. Final model training and future forecasting

In [ ]:
# ============================================================
# FIGURE 11. FINAL MODEL TRAINING AND FUTURE FORECASTING
# ============================================================

W = 6.5
X = W / 2
c = Chart(W, 10.0)
y = 9.95

h = 0.5; API = c.cylinder(X, y - h / 2, 3.2, h, "NASA POWER API",
                          "monthly GHI, cloud amount and RH"); y -= h + GAP

h = 0.46; DL = c.process(X, y - h / 2, 4.6, h, "Download 2005–2025 into a new file",
                         "the original 2005–2024 file is kept unchanged"); y -= h + GAP

h = 0.44; VAL = c.process(X, y - h / 2, 4.6, h, "Validate",
                          "the same checks as data pre-processing"); y -= h + GAP

h = 0.6; DQ = c.decision(X, y - h / 2, 2.0, h, "All checks pass?")

WAIT = c.process(5.42, DQ["y"], 1.66, 0.6, "Wait and retry",
                 "until NASA has\npublished 2025")

y -= h + GAP

h = 0.56; CHK = c.process(X, y - h / 2, 5.6, h, "Check year (2025)",
                          "Retrain on 2005–2024 → forecast 2025 → compare with observations;\n"
                          "its errors are added to the weighting history"); y -= h + GAP

h = 0.46; FIN = c.process(X, y - h / 2, 5.6, h, "Final forecast (2026)",
                          "Retrain on 2005–2025 → forecast January–December 2026 for every grid cell"); y -= h + GAP

h = 0.46; CMB = c.process(X, y - h / 2, 5.6, h, "Combine the components",
                          "time-varying weights from all past errors (2014–2025)"); y -= h + GAP

h = 0.56; SUM = c.process(X, y - h / 2, 5.6, h, "Summarise",
                          "monthly values · annual mean and annual total · difference from the\n"
                          "2005–2025 average · uncertainty ranges from past errors"); y -= h + GAP

h = 0.44; OUT = c.document(X, y - h / 2, 4.2, h, "2026 forecast", "tables (monthly, annual) and maps"); y -= h

c.arrow(bottom(API), top(DL))
c.arrow(bottom(DL), top(VAL))
c.arrow(bottom(VAL), top(DQ))
c.arrow(bottom(DQ), top(CHK))
c.tag(X + 0.08, (bottom(DQ)[1] + top(CHK)[1]) / 2, "Yes")
c.arrow(right(DQ), left(WAIT))
c.tag((right(DQ)[0] + left(WAIT)[0]) / 2, DQ["y"] + 0.09, "No", ha="center")

loop_x = W - 0.18

c.arrow(right(WAIT), (loop_x, WAIT["y"]), (loop_x, DL["y"]), right(DL))

c.arrow(bottom(CHK), top(FIN))
c.arrow(bottom(FIN), top(CMB))
c.arrow(bottom(CMB), top(SUM))
c.arrow(bottom(SUM), top(OUT))

c.legend(y - 0.4, ["cylinder", "process", "decision", "document"])

c.save("future_forecasting_workflow")

plt.show()